## Explainable Healthcare Claims Anomaly Detection

### Project objective
Identify healthcare claims requiring further review using transparent
rules and an Isolation Forest model.

### Data source
Synthetic Synthea data cleaned and validated using PySpark in
Microsoft Fabric. The exported feature table contains one row per claim.

### Approach
1. Load and validate the claim-level features.
2. Apply rules with clear reasons for each flag.
3. Use Isolation Forest to identify unusual claim patterns.
4. Compare rule-based and model-based results.
5. Prepare scored claims for a Streamlit review dashboard.

### Interpretation
This prototype supports retrospective payment and operations review.
Flags indicate review candidates, not confirmed fraud or overpayments.
The dataset does not contain adjudicated fraud labels.

In [12]:
# File paths and financial validation
from pathlib import Path
from decimal import Decimal

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Anomaly detection and preprocessing
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Save and load model artifacts
import joblib

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Load the claim-level feature table

Load the CSV exported from Fabric and confirm that every claim
has a unique, nonmissing ID. Preserve identifiers and clinical codes
as text, and parse the date columns.

In [13]:
from pathlib import Path

project_folder = Path(
    r"C:\Users\konid\Documents\personal\projects for my resume\claims data"
)

print("Notebook working folder:", Path.cwd())

print("\nActual filenames in data:")
for file in (project_folder / "data").iterdir():
    print(repr(file.name))

Notebook working folder: c:\Users\konid\Documents\personal\projects for my resume\claims data\notebooks

Actual filenames in data:
'claims.csv'
'claims_transactions.csv'
'claim_anomaly_features.csv'
'encounters.csv'
'patients.csv'
'payers.csv'
'payer_transitions.csv'


In [14]:
# Locate the CSV from either the project or notebooks folder
candidate_paths = [
    Path("data/claim_anomaly_features.csv"),
    Path("../data/claim_anomaly_features.csv")
]

data_path = next(
    (path for path in candidate_paths if path.is_file()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Cannot find data/claim_anomaly_features.csv. "
        "Check the filename and folder location."
    )

# Keep IDs and clinical codes as text
text_columns = [
    "claim_id",
    "patient_id",
    "encounter_id",
    "provider_id",
    "primary_payer_id",
    "secondary_payer_id",
    "encounter_code",
    *[f"diagnosis_code_{i}" for i in range(1, 9)]
]

claims_features_df = pd.read_csv(
    data_path,
    dtype={column: "string" for column in text_columns},
    parse_dates=["service_date", "birthdate", "deathdate"]
)

# Validate the imported claims
assert len(claims_features_df) == 117889, "Unexpected claim count."
assert claims_features_df["claim_id"].notna().all(), "Missing claim IDs."
assert claims_features_df["claim_id"].is_unique, "Duplicate claim IDs."

print(f"Loaded: {data_path.resolve()}")
print(f"Rows: {claims_features_df.shape[0]:,}")
print(f"Columns: {claims_features_df.shape[1]:,}")

display(claims_features_df.head())

Loaded: C:\Users\konid\Documents\personal\projects for my resume\claims data\data\claim_anomaly_features.csv
Rows: 117,889
Columns: 41


,patient_id,encounter_id,claim_id,provider_id,primary_payer_id,secondary_payer_id,service_date,claim_coverage_category,primary_claim_status,secondary_claim_status,patient_claim_status,total_outstanding_amount,diagnosis_code_1,diagnosis_code_2,diagnosis_code_3,diagnosis_code_4,diagnosis_code_5,diagnosis_code_6,diagnosis_code_7,diagnosis_code_8,diagnosis_count,transaction_count,charge_count,payment_count,transfer_count,total_charge_amount,total_payment_amount,unique_charge_procedure_count,missing_charge_amount_count,missing_payment_amount_count,encounter_class,encounter_code,encounter_duration_minutes,birthdate,deathdate,age_at_service_years,service_before_birth_flag,service_after_death_flag,payment_minus_charge_amount,payment_to_charge_ratio,payment_exceeds_charge_flag
0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,748f8357-6cc7-551d-f31a-32fa2cf84126,e413105d-6f23-34ef-724a-b42adab9df22,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,<NA>,2019-02-17,insured,closed,NaN,closed,0.00,410620009,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,15,3,6,6,877.79,877.79,3,0,0,wellness,410620009,15.00,2019-02-17,NaT,0.00,0,0,0.00,1.00,0
1,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,5a4735ae-423f-6563-28ab-b3d11b49b2d4,8e430d76-6628-c3ac-8950-2acabeb34f86,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,<NA>,2019-03-24,insured,closed,NaN,closed,0.00,410620009,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,10,2,4,4,269.68,269.68,2,0,0,wellness,410620009,15.00,2019-02-17,NaT,0.10,0,0,0.00,1.00,0
2,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,3a69d5f0-26d8-fe66-4a1b-1a7c9ab618c8,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,<NA>,2019-05-26,insured,closed,NaN,closed,0.00,410620009,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,35,7,14,14,"1,292.75","1,292.75",7,0,0,wellness,410620009,15.00,2019-02-17,NaT,0.27,0,0,0.00,1.00,0
3,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,6e93bcf9-45a4-8528-0120-1c1eaa930faf,af33009b-e02a-6959-e5cc-5dffc7398cbd,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,<NA>,2019-07-28,insured,closed,NaN,closed,0.00,410620009,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,35,7,14,14,"1,323.87","1,323.87",7,0,0,wellness,410620009,15.00,2019-02-17,NaT,0.44,0,0,0.00,1.00,0
4,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,8b6787c3-4316-a0cb-899d-4746525c319f,3153e655-a843-2bf1-72eb-1f06a52e524a,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,<NA>,2019-10-27,insured,closed,NaN,closed,0.00,410620009,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,30,6,12,12,831.76,831.76,6,0,0,wellness,410620009,15.00,2019-02-17,NaT,0.69,0,0,0.00,1.00,0


### 2. Inspect feature completeness and existing review flags

Check numeric feature types, missing values, and the review flags
created in Fabric before defining rules and preparing model inputs.

In [15]:
numeric_columns = [
    "diagnosis_count",
    "transaction_count",
    "charge_count",
    "payment_count",
    "transfer_count",
    "total_charge_amount",
    "total_payment_amount",
    "total_outstanding_amount",
    "unique_charge_procedure_count",
    "encounter_duration_minutes",
    "age_at_service_years",
    "payment_minus_charge_amount",
    "payment_to_charge_ratio"
]

feature_check_df = pd.DataFrame({
    "data_type": claims_features_df[numeric_columns].dtypes.astype(str),
    "missing_count": claims_features_df[numeric_columns].isna().sum()
})

display(feature_check_df)

# Confirm existing flag counts
flag_columns = [
    "payment_exceeds_charge_flag",
    "service_before_birth_flag",
    "service_after_death_flag"
]

display(
    claims_features_df[flag_columns]
    .sum()
    .rename("flagged_claim_count")
    .to_frame()
)

# Inspect numeric distributions
display(
    claims_features_df[numeric_columns].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).T
)

,data_type,missing_count
diagnosis_count,int64,0
transaction_count,int64,0
charge_count,int64,0
payment_count,int64,0
transfer_count,int64,0
total_charge_amount,float64,0
total_payment_amount,float64,0
total_outstanding_amount,float64,0
unique_charge_procedure_count,int64,0
encounter_duration_minutes,float64,0


,flagged_claim_count
payment_exceeds_charge_flag,13114
service_before_birth_flag,0
service_after_death_flag,180


,count,mean,std,min,50%,90%,95%,99%,max
diagnosis_count,"117,889.00",1.31,0.65,1.00,1.00,2.00,3.00,4.00,8.00
transaction_count,"117,889.00",6.03,6.73,1.00,4.00,12.00,18.00,28.00,130.00
charge_count,"117,889.00",1.86,2.17,1.00,1.00,4.00,7.00,10.00,28.00
payment_count,"117,889.00",2.14,2.26,0.00,1.00,4.00,5.00,9.00,52.00
transfer_count,"117,889.00",2.04,2.95,0.00,2.00,4.00,8.00,12.00,56.00
total_charge_amount,"117,889.00","1,755.05","6,562.66",0.00,129.16,"2,539.10","13,022.79","26,130.94","625,902.71"
total_payment_amount,"117,889.00","1,759.98","6,561.38",0.00,129.16,"2,539.10","13,022.79","26,130.94","625,902.71"
total_outstanding_amount,"117,889.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
unique_charge_procedure_count,"117,889.00",1.85,2.15,1.00,1.00,4.00,7.00,10.00,24.00
encounter_duration_minutes,"117,889.00",322.76,"37,615.21",15.00,15.00,45.88,133.00,"1,440.00","7,539,840.00"


## 3. Create chronological training and evaluation partitions

Use approximately the earliest 80% of claims for training and the
remaining claims for evaluation. Keep each service date in one partition.

Learn data-derived rule thresholds and model preprocessing from the
training partition only.

This is retrospective analysis of a completed transaction export.
Splitting by service date does not reconstruct which payments were
available at a historical review date.

In [16]:
# Confirm every claim has a valid service date
claims_features_df["service_date"] = pd.to_datetime(
    claims_features_df["service_date"],
    errors="raise"
)

assert claims_features_df["service_date"].notna().all(), (
    "Missing service dates must be investigated before splitting."
)

# Find the service date where cumulative claims reach approximately 80%
claims_per_date = (
    claims_features_df
    .groupby("service_date")
    .size()
    .sort_index()
)

cumulative_share = claims_per_date.cumsum() / claims_per_date.sum()

split_date = cumulative_share[
    cumulative_share >= 0.80
].index[0]

# Keep the entire boundary date in training
train_df = claims_features_df.loc[
    claims_features_df["service_date"] <= split_date
].copy() 

evaluation_df = claims_features_df.loc[
    claims_features_df["service_date"] > split_date
].copy()

assert len(train_df) > 0 and len(evaluation_df) > 0, (
    "Both partitions must contain claims."
)

assert len(train_df) + len(evaluation_df) == len(claims_features_df), (
    "Partition row counts do not reconcile."
)

assert train_df["service_date"].max() < evaluation_df["service_date"].min(), (
    "Training and evaluation dates overlap."
)

partition_summary_df = pd.DataFrame([
    {
        "partition": name,
        "claims": len(df),
        "percentage": round(100 * len(df) / len(claims_features_df), 2),
        "first_service_date": df["service_date"].min().date(),
        "last_service_date": df["service_date"].max().date()
    }
    for name, df in [
        ("Training", train_df),
        ("Evaluation", evaluation_df)
    ]
])

display(partition_summary_df)

# Patients may appear in both chronological partitions
shared_patients = (
    set(train_df["patient_id"].dropna())
    & set(evaluation_df["patient_id"].dropna())
)

print(f"Patients present in both partitions: {len(shared_patients):,}")

,partition,claims,percentage,first_service_date,last_service_date
0,Training,94333,80.02,1912-09-26,2017-11-14
1,Evaluation,23556,19.98,2017-11-15,2021-11-19


Patients present in both partitions: 967


## 4. Apply explainable review rules

Apply financial and date-consistency checks, plus a high-charge
screen using thresholds learned only from training claims.

Calculate high-charge thresholds separately by encounter class.
For classes with fewer than 100 training claims, use the overall
training threshold as a fallback.

These rules identify review candidates, not confirmed errors.

In [17]:
# Learn charge thresholds from training data only
minimum_peer_count = 100

overall_charge_threshold = train_df["total_charge_amount"].quantile(0.99)

peer_charge_thresholds_df = (
    train_df
    .groupby("encounter_class")["total_charge_amount"]
    .agg(
        training_claim_count="size",
        charge_threshold_99=lambda values: values.quantile(0.99)
    )
)

peer_charge_thresholds_df["threshold_used"] = (
    peer_charge_thresholds_df["charge_threshold_99"].where(
        peer_charge_thresholds_df["training_claim_count"]
        >= minimum_peer_count,
        overall_charge_threshold
    )
)

display(peer_charge_thresholds_df)


def apply_review_rules(df):
    result = df.copy()

    # Reuse the financial flag calculated with decimals in Fabric
    result["rule_payment_exceeds_charge"] = (
        result["payment_exceeds_charge_flag"].eq(1)
    )

    result["rule_service_after_death"] = (
        result["service_after_death_flag"].eq(1)
    )

    result["charge_review_threshold"] = (
        result["encounter_class"]
        .map(peer_charge_thresholds_df["threshold_used"])
        .fillna(overall_charge_threshold)
    )

    result["rule_high_charge"] = (
        result["total_charge_amount"]
        > result["charge_review_threshold"]
    )

    rule_columns = [
        "rule_payment_exceeds_charge",
        "rule_service_after_death",
        "rule_high_charge"
    ]

    result["rule_count"] = result[rule_columns].sum(axis=1)
    result["any_rule_flag"] = result[rule_columns].any(axis=1)

    return result


train_rules_df = apply_review_rules(train_df)
evaluation_rules_df = apply_review_rules(evaluation_df)

rule_columns = [
    "rule_payment_exceeds_charge",
    "rule_service_after_death",
    "rule_high_charge",
    "any_rule_flag"
]

rule_summary_df = pd.DataFrame({
    "training_flagged": train_rules_df[rule_columns].sum(),
    "evaluation_flagged": evaluation_rules_df[rule_columns].sum()
})

rule_summary_df["evaluation_percentage"] = (
    100 * rule_summary_df["evaluation_flagged"] / len(evaluation_rules_df)
).round(2)

display(rule_summary_df)

,training_claim_count,charge_threshold_99,threshold_used
encounter_class,,,
ambulatory,22563,"35,142.63","35,142.63"
emergency,2362,"49,138.71","49,138.71"
inpatient,2513,"23,506.46","23,506.46"
outpatient,19582,"15,218.46","15,218.46"
urgentcare,1785,"45,544.35","45,544.35"
wellness,45528,"1,853.91","1,853.91"


,training_flagged,evaluation_flagged,evaluation_percentage
rule_payment_exceeds_charge,11036,2078,8.82
rule_service_after_death,165,15,0.06
rule_high_charge,946,498,2.11
any_rule_flag,12114,2573,10.92


## 5. Prepare Isolation Forest inputs

Use charge amounts and claim activity counts to identify unusual
combinations of billing characteristics.

Apply log(1 + value) to reduce skew while retaining unusually large
values. Fit missing-value imputation on training data only.

Exclude identifiers, demographics, existing rule flags, the constant
outstanding amount, the unstable payment-to-charge ratio, and encounter
duration pending investigation of extreme values.

In [18]:
model_features = [
    "total_charge_amount",
    "diagnosis_count",
    "charge_count",
    "payment_count",
    "transfer_count",
    "unique_charge_procedure_count"
]

# Check that model inputs are numeric, finite when present,
# and nonnegative before applying log1p
for partition_name, df in [
    ("Training", train_rules_df),
    ("Evaluation", evaluation_rules_df)
]:
    values = df[model_features]

    assert all(
        pd.api.types.is_numeric_dtype(values[column])
        for column in model_features
    ), f"{partition_name}: nonnumeric model input."

    assert not np.isinf(values.to_numpy(dtype=float)).any(), (
        f"{partition_name}: infinite model input."
    )

    assert not values.lt(0).any().any(), (
        f"{partition_name}: negative values require investigation."
    )

assert train_rules_df[model_features].notna().any().all(), (
    "A training feature is entirely missing."
)

# Apply the same fixed transformation to both partitions
X_train_log = np.log1p(
    train_rules_df[model_features].astype(float)
)

X_evaluation_log = np.log1p(
    evaluation_rules_df[model_features].astype(float)
)

# Learn replacement values from training data only
model_imputer = SimpleImputer(strategy="median")

X_train = model_imputer.fit_transform(X_train_log)
X_evaluation = model_imputer.transform(X_evaluation_log)

assert np.isfinite(X_train).all(), "Invalid training values remain."
assert np.isfinite(X_evaluation).all(), "Invalid evaluation values remain."

print("Model features:", model_features)
print("Training shape:", X_train.shape)
print("Evaluation shape:", X_evaluation.shape)

display(
    pd.DataFrame(
        X_train,
        columns=model_features,
        index=train_rules_df.index
    ).head()
)

Model features: ['total_charge_amount', 'diagnosis_count', 'charge_count', 'payment_count', 'transfer_count', 'unique_charge_procedure_count']
Training shape: (94333, 6)
Evaluation shape: (23556, 6)


,total_charge_amount,diagnosis_count,charge_count,payment_count,transfer_count,unique_charge_procedure_count
14,5.60,0.69,1.10,1.10,1.61,1.10
15,5.96,1.10,1.10,1.10,1.61,1.10
16,2.15,1.10,0.69,0.69,1.10,0.69
17,5.60,0.69,1.10,1.10,1.61,1.10
18,4.87,0.69,0.69,0.69,1.10,0.69


## 6. Fit Isolation Forest and score claims

Train Isolation Forest on earlier claims and score both partitions.

Use the 99th percentile of training anomaly scores as an initial
review threshold. Higher scores indicate more unusual claims.

This threshold is a prototype review setting, not an estimated fraud
rate. The percentage flagged in evaluation may differ from 1%.

In [19]:
# Train on earlier claims only
isolation_forest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(X_train)

# Negate score_samples so higher scores mean more unusual
train_anomaly_scores = -isolation_forest.score_samples(X_train)

evaluation_anomaly_scores = (
    -isolation_forest.score_samples(X_evaluation)
)

# Set the review threshold using training scores only
model_score_threshold = np.quantile(
    train_anomaly_scores,
    0.99
)

# Keep rules and model results together
train_scored_df = train_rules_df.copy()
evaluation_scored_df = evaluation_rules_df.copy()

train_scored_df["model_anomaly_score"] = train_anomaly_scores
evaluation_scored_df["model_anomaly_score"] = (
    evaluation_anomaly_scores
)

train_scored_df["model_anomaly_flag"] = (
    train_scored_df["model_anomaly_score"]
    > model_score_threshold
)

evaluation_scored_df["model_anomaly_flag"] = (
    evaluation_scored_df["model_anomaly_score"]
    > model_score_threshold
)

model_summary_df = pd.DataFrame([
    {
        "partition": name,
        "claims": len(df),
        "model_flagged": int(df["model_anomaly_flag"].sum()),
        "flagged_percentage": round(
            100 * df["model_anomaly_flag"].mean(),
            2
        )
    }
    for name, df in [
        ("Training", train_scored_df),
        ("Evaluation", evaluation_scored_df)
    ]
])

print(f"Training-derived score threshold: {model_score_threshold:.6f}")
display(model_summary_df)

# Inspect the highest-scoring evaluation claims
display(
    evaluation_scored_df
    .sort_values("model_anomaly_score", ascending=False)
    .head(10)[
        [
            "claim_id",
            "encounter_class",
            *model_features,
            "model_anomaly_score",
            "model_anomaly_flag",
            "any_rule_flag"
        ]
    ]
)

Training-derived score threshold: 0.676036


,partition,claims,model_flagged,flagged_percentage
0,Training,94333,944,1.00
1,Evaluation,23556,597,2.53


,claim_id,encounter_class,total_charge_amount,diagnosis_count,charge_count,payment_count,transfer_count,unique_charge_procedure_count,model_anomaly_score,model_anomaly_flag,any_rule_flag
13587,82401733-60cf-7ad2-31f3-cb47814f8079,ambulatory,"67,649.79",4,23,22,42,23,0.80,True,True
99559,c05a6ebd-b6bc-666c-f0a9-0e692d6ba4d3,ambulatory,"71,689.69",3,21,43,42,21,0.79,True,True
8378,3d0ec425-947c-10af-40d9-50242cfa644f,ambulatory,"67,171.64",3,22,43,42,22,0.79,True,True
26756,a6ca2e62-a33c-7d91-ea1a-a8fe0dc0041f,ambulatory,"76,339.49",3,21,22,42,21,0.79,True,True
30944,65f3f4cf-af94-de85-4247-b4b926a46a05,ambulatory,"72,959.26",3,21,22,42,21,0.79,True,True
24801,91ea5623-27c4-3cb3-c6a9-c2ff9206eccb,ambulatory,"69,292.45",3,24,21,42,24,0.79,True,True
86294,3485185f-4862-536d-9f30-c5e1b3c2c6d6,ambulatory,"61,130.48",3,21,21,42,21,0.79,True,True
98261,55e3a683-1c89-3536-2f00-61244e5597e8,ambulatory,"62,581.41",3,21,22,42,21,0.79,True,True
40940,7c15e7dc-7bbc-34a9-8a49-5249fa8eb0e4,ambulatory,"62,646.83",3,21,22,42,21,0.79,True,True
47592,4a59dc75-e68c-b0c2-46f2-8a402cf5e695,ambulatory,"66,100.32",3,21,22,42,21,0.79,True,True


## 7. Compare rules and Isolation Forest

Separate evaluation claims into four review categories:
rules only, model only, both, and neither.

Summarize the overlap and examine model flagging rates by encounter
class. These results describe review candidates, not confirmed errors
or detection accuracy.

In [20]:
# Rule and model flags for evaluation claims
rule_flag = evaluation_scored_df["any_rule_flag"]
model_flag = evaluation_scored_df["model_anomaly_flag"]

# Assign each claim to exactly one category
evaluation_scored_df["review_category"] = np.select(
    [
        rule_flag & model_flag,
        rule_flag & ~model_flag,
        ~rule_flag & model_flag
    ],
    [
        "Both",
        "Rules only",
        "Model only"
    ],
    default="Neither"
)

category_order = ["Both", "Rules only", "Model only", "Neither"]

comparison_summary_df = (
    evaluation_scored_df["review_category"]
    .value_counts()
    .reindex(category_order, fill_value=0)
    .rename_axis("review_category")
    .to_frame("claim_count")
)

comparison_summary_df["percentage"] = (
    100 * comparison_summary_df["claim_count"]
    / len(evaluation_scored_df)
).round(2)

# Verify that categories account for every evaluation claim
assert (
    comparison_summary_df["claim_count"].sum()
    == len(evaluation_scored_df)
), "Review categories do not reconcile."

assert (
    comparison_summary_df.loc[
        ["Both", "Rules only"], "claim_count"
    ].sum()
    == rule_flag.sum()
), "Rule counts do not reconcile."

assert (
    comparison_summary_df.loc[
        ["Both", "Model only"], "claim_count"
    ].sum()
    == model_flag.sum()
), "Model counts do not reconcile."

display(comparison_summary_df)

# Review flagging rates within each encounter class
encounter_summary_df = (
    evaluation_scored_df
    .groupby("encounter_class")
    .agg(
        claims=("claim_id", "size"),
        rule_flagged=("any_rule_flag", "sum"),
        model_flagged=("model_anomaly_flag", "sum")
    )
)

encounter_summary_df["rule_flagged_percentage"] = (
    100 * encounter_summary_df["rule_flagged"]
    / encounter_summary_df["claims"]
).round(2)

encounter_summary_df["model_flagged_percentage"] = (
    100 * encounter_summary_df["model_flagged"]
    / encounter_summary_df["claims"]
).round(2)

display(encounter_summary_df)

,claim_count,percentage
review_category,,
Both,271,1.15
Rules only,2302,9.77
Model only,326,1.38
Neither,20657,87.69


,claims,rule_flagged,model_flagged,rule_flagged_percentage,model_flagged_percentage
encounter_class,,,,,
ambulatory,5917,665,255,11.24,4.31
emergency,924,101,8,10.93,0.87
inpatient,890,32,84,3.60,9.44
outpatient,6797,815,64,11.99,0.94
urgentcare,791,6,0,0.76,0.00
wellness,8237,954,186,11.58,2.26


## 8. Inspect review candidates

Add an explanation for each triggered rule and inspect representative
claims from each flagged category.

Model-only claims have no triggered rule. Their model scores indicate
unusual patterns but do not explain which features caused the score.

In [22]:
rule_reason_labels = {
    "rule_payment_exceeds_charge":
        "Recorded payments exceed charges by more than $0.01",
    "rule_service_after_death":
        "Service date is after the recorded death date",
    "rule_high_charge":
        "Charge exceeds the encounter-class training threshold"
}

def get_rule_reasons(row):
    reasons = [
        explanation
        for column, explanation in rule_reason_labels.items()
        if row[column]
    ]

    return "; ".join(reasons) if reasons else "No rule triggered"


evaluation_scored_df["rule_reasons"] = (
    evaluation_scored_df.apply(get_rule_reasons, axis=1)
)

review_columns = [
    "claim_id",
    "encounter_class",
    *model_features,
    "total_payment_amount",
    "payment_minus_charge_amount",
    "charge_review_threshold",
    "model_anomaly_score",
    "review_category",
    "rule_reasons"
]

# Show the five highest-scoring claims in each flagged category
for category in ["Both", "Rules only", "Model only"]:
    print(f"\n--- {category} ---")

    examples_df = (
        evaluation_scored_df
        .loc[
            evaluation_scored_df["review_category"].eq(category)
        ]
        .sort_values("model_anomaly_score", ascending=False)
        .head(5)
    )

    display(examples_df[review_columns])


--- Both ---


,claim_id,encounter_class,total_charge_amount,diagnosis_count,charge_count,payment_count,transfer_count,unique_charge_procedure_count,total_payment_amount,payment_minus_charge_amount,charge_review_threshold,model_anomaly_score,review_category,rule_reasons
13587,82401733-60cf-7ad2-31f3-cb47814f8079,ambulatory,"67,649.79",4,23,22,42,23,"67,649.79",0.00,"35,142.63",0.80,Both,Charge exceeds the encounter-class training th...
99559,c05a6ebd-b6bc-666c-f0a9-0e692d6ba4d3,ambulatory,"71,689.69",3,21,43,42,21,"71,689.69",0.00,"35,142.63",0.79,Both,Charge exceeds the encounter-class training th...
8378,3d0ec425-947c-10af-40d9-50242cfa644f,ambulatory,"67,171.64",3,22,43,42,22,"67,171.65",0.01,"35,142.63",0.79,Both,Charge exceeds the encounter-class training th...
26756,a6ca2e62-a33c-7d91-ea1a-a8fe0dc0041f,ambulatory,"76,339.49",3,21,22,42,21,"76,339.49",0.00,"35,142.63",0.79,Both,Charge exceeds the encounter-class training th...
24801,91ea5623-27c4-3cb3-c6a9-c2ff9206eccb,ambulatory,"69,292.45",3,24,21,42,24,"69,292.45",0.00,"35,142.63",0.79,Both,Charge exceeds the encounter-class training th...



--- Rules only ---


,claim_id,encounter_class,total_charge_amount,diagnosis_count,charge_count,payment_count,transfer_count,unique_charge_procedure_count,total_payment_amount,payment_minus_charge_amount,charge_review_threshold,model_anomaly_score,review_category,rule_reasons
100291,624aa8c0-1ff8-f8a9-7198-280f33d8fe32,wellness,"31,691.57",1,9,4,0,9,"31,691.57",0.00,"1,853.91",0.68,Rules only,Charge exceeds the encounter-class training th...
110741,0c3aaea1-ae44-7a89-352a-125aabb5dff6,outpatient,"28,208.72",2,9,5,4,9,"28,208.72",0.00,"15,218.46",0.68,Rules only,Charge exceeds the encounter-class training th...
111855,d09f70c2-859c-74fd-ca2d-ec6cfba9317a,wellness,"39,937.60",1,8,5,0,8,"39,937.60",0.00,"1,853.91",0.67,Rules only,Charge exceeds the encounter-class training th...
69999,4d201e72-c92b-dd3e-c8c0-83dab4e76438,outpatient,"22,968.10",1,9,2,0,9,"22,968.10",0.00,"15,218.46",0.67,Rules only,Charge exceeds the encounter-class training th...
89830,683a2eed-2a50-c78b-9d80-366c2db5a17f,wellness,"15,727.61",2,9,5,10,9,"15,727.61",0.00,"1,853.91",0.67,Rules only,Charge exceeds the encounter-class training th...



--- Model only ---


,claim_id,encounter_class,total_charge_amount,diagnosis_count,charge_count,payment_count,transfer_count,unique_charge_procedure_count,total_payment_amount,payment_minus_charge_amount,charge_review_threshold,model_anomaly_score,review_category,rule_reasons
101483,70315484-c4af-a090-d4a2-961bc3d32d86,inpatient,"21,415.62",5,26,27,52,4,"21,415.62",0.00,"23,506.46",0.77,Model only,No rule triggered
71582,fdeb0af0-2a13-848b-3656-78abf1b74726,inpatient,"20,439.82",5,26,26,52,4,"20,439.82",0.00,"23,506.46",0.77,Model only,No rule triggered
85348,ace07d95-f601-96d2-91a9-79d53da9514d,inpatient,"19,945.98",5,24,25,48,4,"19,945.98",0.00,"23,506.46",0.77,Model only,No rule triggered
23769,ead09a43-560c-62d8-1fa3-e2e2ca6b3a8f,inpatient,"20,656.96",5,26,27,52,4,"20,656.96",0.00,"23,506.46",0.77,Model only,No rule triggered
12433,1a939e1c-f8bd-2f2d-8b14-fcc98940a7d9,inpatient,"20,650.41",6,24,25,48,4,"20,650.41",0.00,"23,506.46",0.77,Model only,No rule triggered


## Interpretation of Flagged Claims

The rules and Isolation Forest identify different types of review
candidates. A flag does not establish fraud, an overpayment, or a
billing error.

### Rules-only claims

These claims trigger at least one defined rule but do not exceed the
model-score threshold of 0.676036.

The displayed examples exceed their encounter-class charge thresholds.
For example, a wellness claim charged $31,691.57 against a training
99th-percentile threshold of $1,853.91. Its overall billing pattern
did not exceed the model's anomaly-score cutoff.

### Model-only claims

These claims exceed the model-score threshold without triggering
any of the three rules.

One inpatient claim charged $21,415.62, below the inpatient charge
threshold of $23,506.46, and had payments equal to charges. It contained
26 charge entries, 27 payment entries, 52 transfer entries, and four
distinct procedure codes. Its model anomaly score was approximately
0.77.

These characteristics provide context for reviewing its unusual
billing pattern. They are not a formal explanation of which features
caused the score. Repeated procedures and financial transfers may
reflect legitimate care and billing activity.

### Claims flagged by both methods

These claims trigger a rule and exceed the model-score threshold.

For example, an ambulatory claim charged $67,649.79, exceeding its
encounter-class threshold of $35,142.63. Its model score was approximately
0.80, and it contained 23 charge entries, 22 payment entries, and
42 transfers.

Agreement between the methods provides two review signals; it does
not independently confirm an error.

### Evaluation Results and Limitations

Of 23,556 evaluation claims:

- 271 were flagged by both methods.
- 2,302 were flagged by rules only.
- 326 were flagged by the model only.
- 20,657 were flagged by neither method.

Together, the methods flagged 2,899 claims (12.31%). Isolation Forest
added 326 candidates beyond the rules.

The model flagged 9.44% of inpatient claims, compared with 2.53% of
evaluation claims overall. This concentration warrants further review
because the global model may identify legitimate inpatient complexity.

Scores displayed to two decimal places are rounded; classification
uses full-precision scores. Flagging rates measure review volume,
not detection accuracy. Claims flagged by neither method are not
guaranteed to be error-free.

## 9. Compare model-only examples with training peers

Compare original feature values with medians from training claims
of the same encounter class. These comparisons provide review context,
not feature-attribution explanations of the model score.

In [23]:
# Learn peer medians from training claims only
training_peer_medians_df = (
    train_rules_df
    .groupby("encounter_class")[model_features]
    .median()
)

# Highest-scoring model-only evaluation claim
example_claim = (
    evaluation_scored_df
    .loc[
        evaluation_scored_df["review_category"].eq("Model only")
    ]
    .sort_values("model_anomaly_score", ascending=False)
    .iloc[0]
)

encounter_class = example_claim["encounter_class"]

peer_comparison_df = pd.DataFrame({
    "claim_value": example_claim[model_features].astype(float),
    "training_peer_median":
        training_peer_medians_df.loc[encounter_class]
})

print("Claim:", example_claim["claim_id"])
print("Encounter class:", encounter_class)
print(
    "Model score:",
    f"{example_claim['model_anomaly_score']:.6f}"
)
print("Model threshold:", f"{model_score_threshold:.6f}")

display(peer_comparison_df)

Claim: 70315484-c4af-a090-d4a2-961bc3d32d86
Encounter class: inpatient
Model score: 0.773027
Model threshold: 0.676036


,claim_value,training_peer_median
total_charge_amount,"21,415.62","1,303.66"
diagnosis_count,5.00,1.00
charge_count,26.00,1.00
payment_count,27.00,1.00
transfer_count,52.00,0.00
unique_charge_procedure_count,4.00,1.00


### Peer Comparison: Model-Only Inpatient Claim

Claim `70315484-c4af-a090-d4a2-961bc3d32d86` received an anomaly
score of 0.773027, exceeding the model review threshold of 0.676036.

Its charge of $21,415.62 was below the inpatient high-charge threshold
of $23,506.46, and recorded payments equaled charges. None of the
three rules triggered.

Compared with training inpatient medians, this claim had substantially
higher charges and billing activity: five populated diagnosis slots,
26 charge entries, 27 payment entries, and 52 transfers. It contained
four distinct procedure codes.

This comparison provides context for reviewing the model-only flag.
It does not establish which features caused the model score.
Multiple charges for the same procedure code and responsibility
transfers may reflect legitimate care and billing workflows.

Further review would examine the underlying procedures and transaction
relationships before determining whether any activity was incorrect.

## 10. Controlled Synthetic Checks

Create altered copies of evaluation claims to check whether the
screening methods respond as expected.

Test a payment discrepancy, service after death, and unusually large
billing activity. These are engineering checks, not estimates of
real-world accuracy. The model is not retrained.

In [24]:
# Select one evaluation claim flagged by neither method
baseline = (
    evaluation_scored_df
    .loc[evaluation_scored_df["review_category"].eq("Neither")]
    .sort_values("claim_id")
    .iloc[[0]]
    .copy()
)

baseline["test_case"] = "Unchanged baseline"

# Test 1: introduce a payment discrepancy
payment_test = baseline.copy()
payment_test["test_case"] = "Payment exceeds charge by $500"
payment_test["total_payment_amount"] = (
    payment_test["total_charge_amount"] + 500
)
payment_test["payment_minus_charge_amount"] = 500.0
payment_test["payment_exceeds_charge_flag"] = 1

# Test 2: put the service date after the recorded death date
date_test = baseline.copy()
date_test["test_case"] = "Service after recorded death"
date_test["deathdate"] = (
    date_test["service_date"] - pd.Timedelta(days=30)
)
date_test["service_after_death_flag"] = 1

# Test 3: create extreme but internally consistent billing counts
activity_test = baseline.copy()
activity_test["test_case"] = "Extreme billing activity"
activity_test["total_charge_amount"] = 1_000_000.0
activity_test["total_payment_amount"] = 1_000_000.0
activity_test["payment_minus_charge_amount"] = 0.0
activity_test["payment_exceeds_charge_flag"] = 0

activity_test["diagnosis_count"] = 8
activity_test["charge_count"] = 100
activity_test["payment_count"] = 100
activity_test["transfer_count"] = 200
activity_test["transaction_count"] = 400
activity_test["unique_charge_procedure_count"] = 50

# Combine test copies and apply the existing rules
synthetic_checks_df = pd.concat(
    [baseline, payment_test, date_test, activity_test],
    ignore_index=True
)

synthetic_checks_df = apply_review_rules(synthetic_checks_df)

# Apply the existing preprocessing and trained model
X_checks_log = np.log1p(
    synthetic_checks_df[model_features].astype(float)
)

X_checks = model_imputer.transform(X_checks_log)

synthetic_checks_df["model_anomaly_score"] = (
    -isolation_forest.score_samples(X_checks)
)

synthetic_checks_df["model_anomaly_flag"] = (
    synthetic_checks_df["model_anomaly_score"]
    > model_score_threshold
)

# Deterministic rule checks
assert synthetic_checks_df.loc[
    1, "rule_payment_exceeds_charge"
], "Payment rule did not trigger."

assert synthetic_checks_df.loc[
    2, "rule_service_after_death"
], "After-death rule did not trigger."

assert synthetic_checks_df.loc[
    3, "rule_high_charge"
], "High-charge rule did not trigger."

display(
    synthetic_checks_df[
        [
            "test_case",
            "rule_payment_exceeds_charge",
            "rule_service_after_death",
            "rule_high_charge",
            "model_anomaly_score",
            "model_anomaly_flag"
        ]
    ]
)

,test_case,rule_payment_exceeds_charge,rule_service_after_death,rule_high_charge,model_anomaly_score,model_anomaly_flag
0,Unchanged baseline,False,False,False,0.55,False
1,Payment exceeds charge by $500,True,False,False,0.55,False
2,Service after recorded death,False,True,False,0.55,False
3,Extreme billing activity,False,False,True,0.81,True


### Interpretation of Controlled Synthetic Checks

We tested an unchanged evaluation claim and three altered copies.
The original dataset and trained model were not modified.

- **Unchanged baseline:** No rules triggered. Its model score was
  approximately 0.55, below the review threshold of 0.676036.

- **Payment discrepancy:** Increasing payments to exceed charges by
  $500 and updating the corresponding flag triggered the payment rule.
  The model score remained unchanged because payment amount is not
  one of its six input features.

- **Service after death:** Setting the recorded death date before the
  service date and updating the corresponding flag triggered the
  after-death rule. The model score remained unchanged because these
  dates are not model inputs.

- **Extreme billing activity:** Increasing the charge amount and billing
  activity counts triggered the high-charge rule. The model score rose
  to approximately 0.81, exceeding the review threshold and triggering
  the model flag.

These results demonstrate complementary screening behavior: rules
identify specified conditions, while Isolation Forest responds to
unusual combinations of its input features.

These are limited engineering checks, not estimates of real-world
accuracy. The payment and date flags were manually updated, so these
tests do not independently validate the upstream Fabric calculations.
A model flag does not establish fraud or a billing error.

## 11. Save Model Artifacts and Review Results

Save the trained Isolation Forest, fitted imputer, feature order,
transformation details, and training-derived thresholds.

Export scored evaluation claims and summary tables for the Streamlit
prototype. Keep controlled synthetic checks in a separate file.

In [25]:
import json
import sklearn

# Find the project folder using the CSV path established earlier
project_root = data_path.resolve().parent.parent

models_folder = project_root / "models"
outputs_folder = project_root / "outputs"

models_folder.mkdir(parents=True, exist_ok=True)
outputs_folder.mkdir(parents=True, exist_ok=True)

# Bundle the fitted model and everything needed to reproduce scoring
model_bundle = {
    "model": isolation_forest,
    "imputer": model_imputer,
    "feature_columns": model_features,
    "transformation": "log1p",
    "score_method": "negative_score_samples",
    "score_threshold": float(model_score_threshold),
    "score_quantile": 0.99,
    "split_date": pd.Timestamp(split_date).isoformat(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}

model_path = models_folder / "isolation_forest_bundle.joblib"

joblib.dump(model_bundle, model_path)

# Save rule thresholds separately
rule_config = {
    "payment_difference_tolerance": 0.01,
    "charge_quantile": 0.99,
    "minimum_peer_count": int(minimum_peer_count),
    "overall_charge_threshold": float(overall_charge_threshold),
    "encounter_charge_thresholds": {
        str(encounter_class): float(threshold)
        for encounter_class, threshold in (
            peer_charge_thresholds_df["threshold_used"].items()
        )
    }
}

with open(
    models_folder / "rule_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(rule_config, file, indent=2)

# Export only the columns needed for the review prototype
export_columns = list(dict.fromkeys([
    "claim_id",
    "service_date",
    "encounter_class",
    *model_features,
    "total_payment_amount",
    "payment_minus_charge_amount",
    "charge_review_threshold",
    "rule_payment_exceeds_charge",
    "rule_service_after_death",
    "rule_high_charge",
    "rule_count",
    "any_rule_flag",
    "rule_reasons",
    "model_anomaly_score",
    "model_anomaly_flag",
    "review_category"
]))

evaluation_scored_df[export_columns].to_csv(
    outputs_folder / "evaluation_scored_claims.csv",
    index=False
)

# Save summaries and training peer context
comparison_summary_df.to_csv(
    outputs_folder / "review_category_summary.csv"
)

encounter_summary_df.to_csv(
    outputs_folder / "encounter_review_summary.csv"
)

training_peer_medians_df.to_csv(
    outputs_folder / "training_peer_medians.csv"
)

peer_charge_thresholds_df.to_csv(
    outputs_folder / "training_charge_thresholds.csv"
)

# Keep artificial examples separate from actual evaluation results
synthetic_checks_df[
    [
        "test_case",
        "rule_payment_exceeds_charge",
        "rule_service_after_death",
        "rule_high_charge",
        "model_anomaly_score",
        "model_anomaly_flag"
    ]
].to_csv(
    outputs_folder / "synthetic_check_results.csv",
    index=False
)

# Verify that the saved model reproduces existing scores
saved_bundle = joblib.load(model_path)

reloaded_scores = -saved_bundle["model"].score_samples(
    X_evaluation[:100]
)

np.testing.assert_allclose(
    reloaded_scores,
    evaluation_scored_df["model_anomaly_score"]
    .iloc[:100]
    .to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

saved_claims_df = pd.read_csv(
    outputs_folder / "evaluation_scored_claims.csv"
)

assert len(saved_claims_df) == len(evaluation_scored_df), (
    "Exported claim count does not match."
)

assert saved_claims_df["claim_id"].is_unique, (
    "Exported claim IDs are not unique."
)

print("Model saved:", model_path)
print(f"Evaluation claims exported: {len(saved_claims_df):,}")
print("Saved model reproduces the checked scores.")
print("Output folder:", outputs_folder)

Model saved: C:\Users\konid\Documents\personal\projects for my resume\claims data\models\isolation_forest_bundle.joblib
Evaluation claims exported: 23,556
Saved model reproduces the checked scores.
Output folder: C:\Users\konid\Documents\personal\projects for my resume\claims data\outputs
